# Multimodal Experiment Runner

This notebook runs a complete end-to-end experiment unattended on Colab. It builds the multimodal dataset, checks modality independence, trains ablation variants via purged walk-forward CV, runs a portfolio backtest, and saves all artifacts to Google Drive.

*(For a pedagogical breakdown of the architecture and data pipeline, see the demo notebook instead.)*


In [ ]:
import os
if not os.path.exists("scripts/run_real_world_demo.py"):
    # If opening directly in Colab, clone the repository first.
    !git clone https://github.com/Randhir123/nifty50-multimodal-transformer.git
    %cd nifty50-multimodal-transformer

!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print("Not running in Colab. Using local path.")
    IN_COLAB = False

In [ ]:
# === EXPERIMENT CONFIG ===
# Edit these. Cell below picks up changes automatically.

TICKERS = "RELIANCE.NS,TCS.NS,INFY.NS,HDFCBANK.NS,ICICIBANK.NS,SBIN.NS"
# Comma-separated NSE tickers. Default is the 6-stock smoke set.
# For full Nifty 50 expansion, paste the full list here.

PERIOD = "1y"
# yfinance period string. "1y", "2y", "5y", "10y", or "max".
# Longer periods need more compute and exercise more market regimes.

WINDOW_SIZE = 20
HORIZON_DAYS = 3
# OHLCV window length and forward-return horizon. Don't change unless
# you know why — these match the leakage-test assumptions.

EPOCHS = 20
BATCH_SIZE = 16
# Training schedule per ablation variant per fold.

CV_SPLITS = 3
EMBARGO_DAYS = 3
# Walk-forward purged CV.

OUTPUT_ROOT = "/content/drive/MyDrive/nifty50_experiments" if IN_COLAB else "./experiments"
# Where artifacts are written. A timestamped subdirectory is created.

DEVICE = "cuda" # set to "cpu" if no GPU

In [ ]:
import os
import time
import traceback
import json
from pathlib import Path
import torch
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Hard-coded configuration (determined in session 5, do not change)
SEED = 42
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
LOSS_FUNCTION = "BCEWithLogitsLoss"
OPTIMIZER = "AdamW"
# Note: Multi-seed evaluation is out of scope for this unattended runner.

# Set up paths
timestamp = time.strftime("%Y%m%d_%H%M%S")
run_id = f"exp_{timestamp}"
output_dir = Path(OUTPUT_ROOT) / run_id
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Run ID: {run_id}")
print(f"Output Directory: {output_dir}")

# Initialize summary content
summary_content = [
    f"# Experiment Summary: {run_id}\n",
    "## Configuration",
    f"- **Tickers**: {len([t for t in TICKERS.split(',') if t.strip()])} stocks",
    f"- **Period**: {PERIOD}",
    f"- **Window Size**: {WINDOW_SIZE}",
    f"- **Horizon Days**: {HORIZON_DAYS}",
    f"- **Epochs**: {EPOCHS}",
    f"- **Batch Size**: {BATCH_SIZE}",
    f"- **CV Splits**: {CV_SPLITS}",
    f"- **Embargo Days**: {EMBARGO_DAYS}",
    f"- **Device**: {DEVICE}\n",
    "## Execution Log"
]
start_time = time.time()

In [ ]:
import sys
import subprocess
import yfinance as yf
from datetime import date, timedelta

print("Step 1: Pre-downloading data with retries...")
raw_dir = output_dir / "raw"
raw_dir.mkdir(exist_ok=True)

ticker_list = [t.strip() for t in TICKERS.split(",") if t.strip()]
end_date = date.today()

if PERIOD.endswith("mo"):
    start_date = end_date - timedelta(days=int(PERIOD[:-2]) * 31)
elif PERIOD.endswith("y"):
    start_date = end_date - timedelta(days=int(PERIOD[:-1]) * 365)
else:
    start_date = end_date - timedelta(days=365)

start_date_str = start_date.isoformat()
end_date_str = end_date.isoformat()

successful_tickers = []
for ticker in ticker_list:
    success = False
    for attempt in range(3):
        try:
            df = yf.download(ticker, start=start_date_str, end=end_date_str, progress=False)
            if not df.empty:
                df.to_csv(raw_dir / f"{ticker.replace('.', '_')}.csv")
                successful_tickers.append(ticker)
                success = True
                break
        except Exception as e:
            time.sleep(2 ** attempt)
    if not success:
        print(f"Failed to download {ticker} after 3 attempts.")

for attempt in range(3):
    try:
        df_bench = yf.download("^NSEI", start=start_date_str, end=end_date_str, progress=False)
        if not df_bench.empty:
            df_bench.to_csv(raw_dir / "NSEI.csv")
            break
    except:
        time.sleep(2 ** attempt)

if len(successful_tickers) < len(ticker_list) / 2:
    raise RuntimeError("Fewer than half of the tickers downloaded successfully. Aborting.")

print("\nStep 1.b: Building Multimodal Samples via pipeline script...")
dataset_path = output_dir / "real_world_multimodal_samples.npz"
try:
    subprocess.run([
        sys.executable, "scripts/run_real_world_demo.py",
        "--tickers", *successful_tickers,
        "--period", PERIOD,
        "--window-size", str(WINDOW_SIZE),
        "--horizon-days", str(HORIZON_DAYS),
        "--output-dir", str(output_dir),
        "--device", DEVICE
    ], check=True)
    print(f"Saved multimodal artifact to {dataset_path}")
    summary_content.append("- Data Prep: SUCCESS")
except Exception as e:
    print(f"Error during data prep: {e}")
    traceback.print_exc()
    summary_content.append(f"- Data Prep: FAILED ({e})")


In [ ]:
if dataset_path.exists():
    data = np.load(dataset_path, allow_pickle=False)
    num_samples = len(data["y"])
    pos_rate = data["y"].sum() / num_samples if num_samples > 0 else 0
    
    print(f"Total Samples: {num_samples}")
    print(f"Positive Label Rate: {pos_rate:.1%}")
    for k in data.files:
        if data[k].ndim > 0:
            print(f"  {k}: {data[k].shape}")
    
    summary_content.append("\n## Artifact Summary")
    summary_content.append(f"- Total Samples: {num_samples}")
    summary_content.append(f"- Positive Rate: {pos_rate:.1%}")
    for k in data.files:
        if data[k].ndim > 0:
            summary_content.append(f"- `{k}`: {data[k].shape}")
    del data


In [ ]:
print("Step 2: Checking Modality Independence...")
try:
    if dataset_path.exists():
        indep_csv = output_dir / "modality_independence.csv"
        subprocess.run([
            sys.executable, "scripts/check_modality_independence.py",
            "--artifact", str(dataset_path),
            "--output-csv", str(indep_csv)
        ], check=True)
        
        if indep_csv.exists():
            df_indep = pd.read_csv(indep_csv, index_col=0)
            display(df_indep)
            
            summary_content.append("\n## Modality Independence")
            summary_content.append(df_indep.to_markdown())
            summary_content.append("\n- Independence Check: SUCCESS")
except Exception as e:
    print(f"Error during independence check: {e}")
    summary_content.append(f"- Independence Check: FAILED ({e})")


In [ ]:
print("Step 3: Running Ablations (this will take a while)...")
try:
    if dataset_path.exists():
        # Known gap: The ablation runner does not support resuming from a mid-fold crash natively.
        subprocess.run([
            sys.executable, "scripts/run_ablation_study.py",
            "--dataset", str(dataset_path),
            "--output-dir", str(output_dir / "ablations"),
            "--epochs", str(EPOCHS),
            "--batch-size", str(BATCH_SIZE),
            "--device", DEVICE,
            "--cv-splits", str(CV_SPLITS),
            "--horizon-days", str(HORIZON_DAYS),
            "--embargo-days", str(EMBARGO_DAYS)
        ], check=True)
        summary_content.append("- Ablations: SUCCESS")
except Exception as e:
    print(f"Error during ablations: {e}")
    summary_content.append(f"- Ablations: FAILED ({e})")


In [ ]:
try:
    ablation_csv = output_dir / "ablations" / "ablation_results.csv"
    if ablation_csv.exists():
        df_ablation = pd.read_csv(ablation_csv)
        cols = [c for c in ["variant", "accuracy_mean", "roc_auc_mean", "f1_mean", "val_positive_rate"] if c in df_ablation.columns]
        display(df_ablation[cols])
        
        summary_content.append("\n## Ablation Results")
        summary_content.append(df_ablation[cols].to_markdown(index=False))
except Exception as e:
    print(f"Error displaying ablations: {e}")


In [ ]:
print("Step 4: Running Corrected Backtest...")
try:
    # Prefer tabular_image as the best performing single addition (from session 7), 
    # fallback to tabular_only if not available.
    preds_csv = output_dir / "ablations" / "prediction_scores_tabular_image.csv"
    if not preds_csv.exists():
        preds_csv = output_dir / "ablations" / "prediction_scores_tabular_only.csv"
        
    tab_csv = output_dir / "tabular_samples.csv"
    
    if preds_csv.exists() and tab_csv.exists():
        subprocess.run([
            sys.executable, "scripts/run_backtest.py",
            "--predictions", str(preds_csv),
            "--tabular-samples", str(tab_csv),
            "--raw-dir", str(raw_dir),
            "--output-dir", str(output_dir / "backtest"),
            "--top-k", "1",
            "--horizon-days", str(HORIZON_DAYS)
        ], check=True)
        
        img_path = output_dir / "backtest" / "backtest_curve.png"
        if img_path.exists():
            display(Image(filename=str(img_path)))
        
        metrics_path = output_dir / "backtest" / "backtest_metrics.json"
        if metrics_path.exists():
            with open(metrics_path, "r") as f:
                metrics = json.load(f)
            summary_content.append("\n## Backtest Headline Numbers")
            for k, v in metrics.items():
                summary_content.append(f"- **{k}**: {v}")
        
        summary_content.append("\n- Backtest: SUCCESS")
except Exception as e:
    print(f"Error during backtest: {e}")
    summary_content.append(f"- Backtest: FAILED ({e})")


In [ ]:
end_time = time.time()
elapsed = (end_time - start_time) / 60
summary_content.append(f"\n**Total Runtime**: {elapsed:.2f} minutes")

summary_file = output_dir / "summary.md"
with open(summary_file, "w") as f:
    f.write("\n".join(summary_content))

print(f"\nDone! Experiment completed in {elapsed:.2f} minutes.")
print(f"All results saved to: {output_dir}")
display(Markdown(f"Saved summary to `{summary_file}`"))
